In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    count,
    current_timestamp,
    lit,
    lower,
    lpad,
    month,
    round as spark_round,
    sum as spark_sum,
    to_date,
    trim,
    when,
    year
)

from pyspark.sql.window import Window

SILVER_RASTREAMENTO_TABLE = "ecommerce_rastreamento_entregas"
SILVER_FERIADOS_TABLE = "feriados"

SILVER_RASTREAMENTO_PATH = f"{SILVER_BASE_PATH}{SILVER_RASTREAMENTO_TABLE}"
SILVER_FERIADOS_PATH = f"{SILVER_BASE_PATH}{SILVER_FERIADOS_TABLE}"

GOLD_TABLE = "gold_ecommerce_rastreamento_entregas_feriados_ocorrencias"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

RASTREAMENTO_REQUIRED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao"
]

FERIADOS_REQUIRED_COLUMNS = [
    "data_feriado"
]

GOLD_KEY_COLUMNS = [
    "ano_evento",
    "mes_evento",
    "id_transportadora",
    "evento_em_feriado",
    "grupo_ocorrencia",
    "tipo_ocorrencia"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
df_rastreamento = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_RASTREAMENTO_PATH)
)

df_feriados = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_FERIADOS_PATH)
)

validate_required_columns(df_rastreamento, RASTREAMENTO_REQUIRED_COLUMNS)
validate_required_columns(df_feriados, FERIADOS_REQUIRED_COLUMNS)

total_rastreamento = df_rastreamento.count()

df_feriados_base = (
    df_feriados
    .select(to_date(col("data_feriado")).alias("data_evento"))
    .distinct()
)

df_eventos_base = (
    df_rastreamento
    .select(
        col("id_rastreamento").cast("int").alias("id_rastreamento"),
        col("id_pedido_ecommerce").cast("int").alias("id_pedido_ecommerce"),
        col("id_transportadora").cast("int").alias("id_transportadora"),
        col("status_entrega"),
        col("dt_evento"),
        to_date(col("dt_evento")).alias("data_evento"),
        lower(trim(col("observacao"))).alias("observacao_normalizada")
    )
    .withColumn("ano_evento", year(col("dt_evento")))
    .withColumn("mes_evento", month(col("dt_evento")))
)

df_eventos_feriados = (
    df_eventos_base
    .join(
        df_feriados_base.withColumn("evento_em_feriado", lit(1)),
        on="data_evento",
        how="left"
    )
    .withColumn(
        "evento_em_feriado",
        when(col("evento_em_feriado").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "tipo_ocorrencia",
        when(col("observacao_normalizada").isNull(), "sem_observacao")
        .when(col("observacao_normalizada").contains("problemas na malha"), "problema_malha_logistica")
        .when(
            col("observacao_normalizada").contains("porteiro") |
            col("observacao_normalizada").contains("vizinho"),
            "recebido_porteiro_vizinho"
        )
        .when(col("observacao_normalizada").contains("entrega realizada para terceiro"), "entrega_terceiro")
        .otherwise("outra_observacao")
    )
    .withColumn(
        "grupo_ocorrencia",
        when(col("tipo_ocorrencia") == "sem_observacao", "sem_observacao")
        .when(col("tipo_ocorrencia") == "problema_malha_logistica", "problema_logistico")
        .when(
            col("tipo_ocorrencia").isin("recebido_porteiro_vizinho", "entrega_terceiro"),
            "entrega_com_ressalva"
        )
        .otherwise("outra_observacao")
    )
)

total_eventos_apos_join = df_eventos_feriados.count()

print(f"Total eventos rastreamento: {total_rastreamento}")
print(f"Total eventos após join com feriados: {total_eventos_apos_join}")

if total_eventos_apos_join != total_rastreamento:
    raise Exception("Erro: o join com feriados alterou a quantidade de eventos.")

print("Base classificada com sucesso.")
display(
    df_eventos_feriados
    .groupBy("evento_em_feriado", "grupo_ocorrencia", "tipo_ocorrencia")
    .count()
    .orderBy("evento_em_feriado", col("count").desc())
)

In [0]:
from pyspark.sql.functions import col, count, round as spark_round, sum as spark_sum
from pyspark.sql.window import Window

window_feriado = Window.partitionBy("evento_em_feriado")

df_proporcao_ocorrencias = (
    df_eventos_feriados
    .groupBy(
        "evento_em_feriado",
        "grupo_ocorrencia",
        "tipo_ocorrencia"
    )
    .agg(
        count("*").alias("qtd_eventos")
    )
    .withColumn(
        "total_eventos_feriado_ou_nao",
        spark_sum("qtd_eventos").over(window_feriado)
    )
    .withColumn(
        "percentual_eventos",
        spark_round(
            (col("qtd_eventos") / col("total_eventos_feriado_ou_nao")) * 100,
            2
        )
    )
    .orderBy(
        "evento_em_feriado",
        col("percentual_eventos").desc()
    )
)

display(df_proporcao_ocorrencias)

In [0]:
from pyspark.sql.functions import when

df_proporcao_ocorrencias_grafico = (
    df_proporcao_ocorrencias
    .withColumn(
        "tipo_dia",
        when(col("evento_em_feriado") == 1, "Feriado")
        .otherwise("Não feriado")
    )
)

display(df_proporcao_ocorrencias_grafico)